# Aksara OCR — ConvNeXt-Tiny fix

ConvNeXt-Tiny diverged in the backbone run (macro-F1 0.00) because lr=1e-3 is
too high for it. This reruns *only* ConvNeXt at lr=1e-4.

**Setup:** T4 x2, Internet On, **Save & Run All (Commit)**. ~3 runs x ~40 min
≈ 2 h — fits one session.

In [1]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Settings (right sidebar) > Accelerator > GPU T4 x2, then rerun."
)

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
arch = f"sm_{major}{minor}"
supported = [a for a in torch.cuda.get_arch_list() if a.startswith("sm_")]

print(f"{name}  ({arch})")
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM")
print(f"torch {torch.__version__}  supports: {supported}")

# torch.cuda.is_available() returns True even when this build ships no kernels
# for the device - the failure then surfaces as a warning storm with every run
# landing in failures.jsonl. Check the architecture explicitly and stop here.
if arch not in supported:
    raise SystemExit(
        f"{name} is {arch}, but this PyTorch build only has kernels for "
        f"{supported}. Switch Settings > Accelerator to GPU T4 x2 (sm_75) "
        f"and rerun. The P100 is sm_60 and will not work."
    )

# Prove a real kernel runs, not just that a device is listed.
probe = (torch.randn(512, 512, device="cuda") @ torch.randn(512, 512, device="cuda")).sum()
torch.cuda.synchronize()
print(f"GPU compute OK (probe={probe.item():.1f})")

Tesla T4  (sm_75)
15.6 GB VRAM
torch 2.10.0+cu128  supports: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
GPU compute OK (probe=1014.3)


In [2]:
# Internet must be ON (Settings > Internet) for these three lines.
import os
from pathlib import Path

REPO_URL = "https://github.com/phoenixfin/aksantara-ocr.git"
REPO = Path("/kaggle/working/aksantara-ocr")

if REPO.exists():
    !cd {REPO} && git pull -q
else:
    !git clone -q {REPO_URL} {REPO}

os.chdir(REPO)
# torch/torchvision ship with Kaggle; installing the rest avoids a slow reinstall
# of torch against a possibly-mismatched CUDA build.
!pip install -q timm pyyaml scikit-image tabulate
print(f"ready: {Path.cwd()}")

ready: /kaggle/working/aksantara-ocr


In [3]:
# The diverged ConvNeXt runs wrote result.json files, and the runner skips any
# run whose result.json exists. Delete them so the fixed config actually reruns.
import shutil
from pathlib import Path
ARTIFACTS = Path("/kaggle/working/artifacts")
RESULTS = ARTIFACTS / "results" / "backbones"
RESULTS.mkdir(parents=True, exist_ok=True)

# Restore other finished backbones (so they are not redone) but NOT convnext.
sources  = list(REPO.glob("artifacts_kaggle/**/results/backbones"))
sources += list(Path("/kaggle/input").glob("*/artifacts/results/*"))
restored = purged = 0
for cand in sources:
    if not cand.is_dir(): continue
    for run in cand.iterdir():
        if not (run/"result.json").exists(): continue
        if "convnext" in run.name:   # skip the broken ones on purpose
            continue
        dest = RESULTS/run.name
        if not dest.exists() and "__unified__" in run.name:
            shutil.copytree(run, dest); restored += 1
# also purge any convnext already sitting in the working dir
for run in RESULTS.glob("convnext_tiny__*"):
    shutil.rmtree(run); purged += 1
print(f"restored {restored} non-convnext runs, purged {purged} broken convnext runs")

restored 0 non-convnext runs, purged 0 broken convnext runs


In [4]:
# Fetch the cleaned, published v3. ~808 MB; needs Internet ON.
DOI = "10.17632/vfj32bpjsf.3"
!python scripts/00_fetch_mendeley.py --doi "{DOI}" --list-only
!python scripts/00_fetch_mendeley.py --doi "{DOI}" --out /kaggle/working/raw

RAW_ROOT = "/kaggle/working/raw" 

Dataset vfj32bpjsf v3

13 file(s), 808.3 MB total:

      230.0 MB  Bali.zip
       21.2 MB  Batak.zip
        2.4 MB  Bima.zip
       44.6 MB  Dunging-Iban.zip
      141.9 MB  Jawa.zip
        8.3 MB  Jawi.zip
       22.2 MB  Kawi.zip
       14.1 MB  Lampung.zip
       84.5 MB  Lontara.zip
       48.0 MB  Minangkabau.zip
       44.8 MB  Ogan.zip
       13.7 MB  Pallawa.zip
      132.7 MB  Sunda.zip
Dataset vfj32bpjsf v3

13 file(s), 808.3 MB total:

      230.0 MB  Bali.zip
       21.2 MB  Batak.zip
        2.4 MB  Bima.zip
       44.6 MB  Dunging-Iban.zip
      141.9 MB  Jawa.zip
        8.3 MB  Jawi.zip
       22.2 MB  Kawi.zip
       14.1 MB  Lampung.zip
       84.5 MB  Lontara.zip
       48.0 MB  Minangkabau.zip
       44.8 MB  Ogan.zip
       13.7 MB  Pallawa.zip
      132.7 MB  Sunda.zip

  get : Bali.zip
       230.0 / 230.0 MB  (100.0%)
    extracted -> /kaggle/working/raw

  get : Batak.zip
        21.2 / 21.2 MB  (100.0%)
    extracted -> /kaggle/working/raw

  get : Bima.zi

In [5]:
# Pre-resize once to 224px. Source images run up to 1500x1500; decoding one
# costs ~6.7 ms/core, so without this the dataloader, not the GPU, is the limit.
# A 224px cache drops that to ~1 ms/img.
!python scripts/00b_build_cache.py \
    --data-root "{RAW_ROOT}" --out /kaggle/working/data --size 224

DATA_ROOT = "/kaggle/working/data"
# Free the disk — the full-size tree is not needed again this session.
!rm -rf {RAW_ROOT}

Scanning /kaggle/working/raw ...
97383 image(s); 0 already cached, 97383 to do.
  97383/97383  (565 img/s, ETA 0.0 min)

Cached 97383 image(s) in 2.9 min -> /kaggle/working/data

images: 97383 source -> 97383 cached
size  : 875 MB -> 782 MB

NOTE: 9 image(s) became byte-identical to another at 224px (they differ in the source).
      Downscaling merged them. They are a leakage path if split across train/test —
      pass --drop-duplicates to 01_prepare_data.py to remove them.

Next:
  python scripts/01_prepare_data.py --data-root /kaggle/working/data --drop-duplicates


In [6]:
# stratified (no writer ids); --drop-duplicates removes images that became
# byte-identical after the 224px resize.
!python scripts/01_prepare_data.py \
    --data-root "{DATA_ROOT}" --out-dir "{ARTIFACTS}" \
    --split-strategy stratified --drop-duplicates

# Verify against published v3. Image/class/script counts come from manifest.csv,
# written before any filtering, so they are invariant.
import pandas as pd
manifest = pd.read_csv(f"{ARTIFACTS}/manifest.csv")
EXPECTED = {"images": 97383, "classes": 889, "scripts": 13}
actual = {"images": len(manifest), "classes": manifest["label"].nunique(),
          "scripts": manifest["script"].nunique()}
for k, want in EXPECTED.items():
    print(f"  {k:8} {actual[k]:6}  expected {want:6}  {'OK' if actual[k]==want else 'MISMATCH'}")
if actual != EXPECTED:
    raise SystemExit("Data does not match published v3 — re-run fetch and cache cells.")
print("Matches published v3.")

Scanning /kaggle/working/data ...

  images     : 97383
  scripts    : 13
  characters : 889 (script/character pairs)
  with writer id: 0/97383
  images per character: min=19 median=60 max=694

           Left in place, these can leak across the train/test boundary.
           Written to /kaggle/working/artifacts/duplicates.csv for review.

Manifest -> /kaggle/working/artifacts/manifest.csv

Dropped 9 duplicate image(s); 97374 remain.

Splitting (strategy=stratified, seed=42) ...
{
  "counts": {
    "train": 69552,
    "test": 13911,
    "val": 13911
  },
  "classes_per_split": {
    "train": 889,
    "val": 889,
    "test": 889
  },
  "total_classes": 889,
  "classes_absent_from_train": []
}

Splits -> /kaggle/working/artifacts/splits.csv

Next: python scripts/02_run_matrix.py --config configs/full_benchmark.yaml
  images    97383  expected  97383  OK
  classes     889  expected    889  OK
  scripts      13  expected     13  OK
Matches published v3.


In [7]:
# Rerun ConvNeXt-Tiny at lr=1e-4.
!python scripts/02_run_matrix.py --config configs/backbone_convnext_fix.yaml     --artifacts "{ARTIFACTS}" --results "{RESULTS}" --num-workers 4 --time-budget 8

Loaded 97374 images, 889 classes, 13 scripts.

Matrix expands to 3 experiments -> /kaggle/working/artifacts/results/backbones
Device: cuda
[1/3] run: convnext_tiny__unified__sz64__aug-medium__pt__s0
    preloading 1.20 GB into memory
preload 64px: 100%|█████████████████████| 13911/13911 [00:13<00:00, 1065.66it/s]
model.safetensors: 100%|█████████████████████| 114M/114M [00:02<00:00, 51.0MB/s]
    acc=0.9904  macro_f1=0.9867
[2/3] run: convnext_tiny__unified__sz64__aug-medium__pt__s1
    preloading 1.20 GB into memory
    acc=0.9910  macro_f1=0.9867
[3/3] run: convnext_tiny__unified__sz64__aug-medium__pt__s2
    preloading 1.20 GB into memory
    acc=0.9917  macro_f1=0.9868

3 run(s) in 1.58h (mean 31.6min/run)

3 runs complete. Tables -> /kaggle/working/artifacts/results/backbones/report

Top 10 by test macro-F1:
  0.9868  convnext_tiny__unified__sz64__aug-medium__pt__s2
  0.9867  convnext_tiny__unified__sz64__aug-medium__pt__s0
  0.9867  convnext_tiny__unified__sz64__aug-medium__pt__s

In [8]:
import json
from pathlib import Path
for f in sorted(Path(RESULTS).glob("convnext_tiny__*/result.json")):
    d=json.load(open(f)); m=d["test_metrics"]
    print(f"{f.parent.name}: acc={m['accuracy']*100:.2f}  macro_f1={m['macro_f1']*100:.2f}")

convnext_tiny__unified__sz64__aug-medium__pt__s0: acc=99.04  macro_f1=98.67
convnext_tiny__unified__sz64__aug-medium__pt__s1: acc=99.10  macro_f1=98.67
convnext_tiny__unified__sz64__aug-medium__pt__s2: acc=99.17  macro_f1=98.68


If macro-F1 is now ~98-99, ConvNeXt is fixed. Save Version to keep the output.